In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

df = pd.read_csv("messy_ecommerce_sales_data.csv")
print(df.head())
print(df.info())
print(df.describe())

    ID  Customer_Name   Order_ID  Order_Date        Product     Category  \
0  100   Customer_100  ORD-41285  11/22/2024        Blender         Home   
1  101   Customer_101  ORD-35783    7/5/2025     Smartphone  Electronics   
2  102   Customer_102  ORD-84355  12/23/2024  Tennis Racket       Sports   
3  103   Customer_103  ORD-57811   3/19/2025        Science        Books   
4  104   Customer_104  ORD-93614  10/20/2025      Biography        Books   

  Quantity   Price    Payment_Method      Status    Total  
0        3      38  Cash on Delivery     Shipped   114.00  
1        2     abd            PayPal  Processing      NaN  
2        1  389.05            PayPal   Delivered   389.05  
3        5  233.92            PayPal  Processing  1169.60  
4        1  552.51  Cash on Delivery  Processing   552.51  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          ------------

In [3]:
#Price fix
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

#Date fix
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

#Columnnames
print(df.columns.tolist())
df["Customer_Name"] = df[" Customer_Name"]
df["Category"] = df[" Category"]
df = df.drop([" Category", " Customer_Name"], axis='columns')

print(df.info())

['ID', ' Customer_Name', 'Order_ID', 'Order_Date', 'Product', ' Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              103 non-null    int64         
 1   Order_ID        103 non-null    object        
 2   Order_Date      101 non-null    datetime64[ns]
 3   Product         103 non-null    object        
 4   Quantity        98 non-null     object        
 5   Price           94 non-null     float64       
 6   Payment_Method  103 non-null    object        
 7   Status          103 non-null    object        
 8   Total           89 non-null     float64       
 9   Customer_Name   103 non-null    object        
 10  Category        95 non-null     object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 9.0+ KB
None


In [4]:
#CustomerName
print(df.groupby("ID").size().sort_values(ascending=False))

print(df[df.groupby("ID").transform('size') > 1])

df["ID"] = range(1, len(df)+1)
print(df.head())


print(df["Customer_Name"].duplicated().sum())

mask = ~df["Customer_Name"].str.match(r"^Customer_\d+$")

print(df[mask])
#Name is consistent 


ID
146    2
175    2
142    2
100    1
104    1
      ..
195    1
196    1
197    1
198    1
199    1
Length: 100, dtype: int64
      ID   Order_ID Order_Date     Product Quantity   Price Payment_Method  \
42   142  ORD-69018 2025-10-30       Shoes        5  645.26    Credit Card   
46   146  ORD-32755 2025-07-09  Basketball        2  705.42  Bank Transfer   
75   175  ORD-56651 2025-02-24  Headphones        1  111.36    Credit Card   
100  175  ORD-56651 2025-02-24  Headphones        1  111.36    Credit Card   
101  142  ORD-69018 2025-10-30       Shoes        5  645.26    Credit Card   
102  146  ORD-32755 2025-07-09  Basketball        2  705.42  Bank Transfer   

         Status     Total Customer_Name     Category  
42      Shipped  2258.410  Customer_142     Clothing  
46   Processing  1410.840  Customer_146   electronic  
75   Processing   111.360  Customer_175  Electronics  
100  Processing    77.952  Customer_175  Electronics  
101     Shipped  3226.300  Customer_142     Clothi

In [5]:
#NaN in Price
count_of_nan = len(df[df["Price"].isna()])
print(9 / 103) 
#8.73% of NaN values

df["Category"] = df["Category"].fillna("Unknown")

#fill with median of category
df["Price"] = df["Price"].fillna(df.groupby("Category")["Price"].transform("median"))
#fill those where category has only one member
df["Price"] = df["Price"].fillna(df["Price"].median())

0.08737864077669903


In [ ]:
#Resolve Total and Quanity 

#Price is not null
#Quantity might be null
#Total might be null
#Total = Price * Quantity
#Quantity = Total / Price

df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

#We need to filter those where Quanity & Total is null
mask = df["Quantity"].isna() & df["Total"].isna()
mask_p = df["Price"] != 0

mask_keep = mask_p & (~mask)
df = df[mask_keep].copy()

#fill quantity where total is not null
mask_q = df["Total"].notna() & df["Quantity"].isna()

df.loc[mask_q, "Quantity"] = df.loc[mask_q, "Total"] / df.loc[mask_q, "Price"]

#fill total, where quantity is not null
mask_t = df["Quantity"].notna() & df["Total"].isna()

df.loc[mask_t, "Total"] = df.loc[mask_t, "Quantity"] *df.loc[mask_t, "Price"]

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 97 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              97 non-null     int64         
 1   Order_ID        97 non-null     object        
 2   Order_Date      96 non-null     datetime64[ns]
 3   Product         97 non-null     object        
 4   Quantity        97 non-null     float64       
 5   Price           97 non-null     float64       
 6   Payment_Method  97 non-null     object        
 7   Status          97 non-null     object        
 8   Total           97 non-null     float64       
 9   Customer_Name   97 non-null     object        
 10  Category        97 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(6)
memory usage: 9.1+ KB
None


In [7]:
#Order date
df["Order_Date"] = df["Order_Date"].ffill
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

print(df.info())


<class 'pandas.core.frame.DataFrame'>
Index: 97 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              97 non-null     int64         
 1   Order_ID        97 non-null     object        
 2   Order_Date      0 non-null      datetime64[ns]
 3   Product         97 non-null     object        
 4   Quantity        97 non-null     float64       
 5   Price           97 non-null     float64       
 6   Payment_Method  97 non-null     object        
 7   Status          97 non-null     object        
 8   Total           97 non-null     float64       
 9   Customer_Name   97 non-null     object        
 10  Category        97 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(6)
memory usage: 9.1+ KB
None


In [8]:
#Outliers in Price IQR Method 
first_quantile = df["Price"].quantile(0.25)
third_quantile = df["Price"].quantile(0.75)

print(first_quantile)
print(third_quantile)

IQR = third_quantile - first_quantile
print(IQR)

upper_bound = third_quantile + 1.5*IQR
lower_bound = first_quantile - 1.5*IQR

df = df[df["Price"].between(lower_bound, upper_bound)]

print(df.info())



275.1
705.42
430.31999999999994
<class 'pandas.core.frame.DataFrame'>
Index: 96 entries, 0 to 102
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ID              96 non-null     int64         
 1   Order_ID        96 non-null     object        
 2   Order_Date      0 non-null      datetime64[ns]
 3   Product         96 non-null     object        
 4   Quantity        96 non-null     float64       
 5   Price           96 non-null     float64       
 6   Payment_Method  96 non-null     object        
 7   Status          96 non-null     object        
 8   Total           96 non-null     float64       
 9   Customer_Name   96 non-null     object        
 10  Category        96 non-null     object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(6)
memory usage: 9.0+ KB
None
